In [6]:
import numpy as np
import pandas as pd
TARGETS = ["Wd", "We"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [7]:
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        for col in [col_r2, col_mse]:
            if col in results.columns:
                results[col] = (
                    results[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )

In [40]:
best_models_tables = {}
best_only = []

summary_all = []     # estatísticas com todos
summary_topN = []   # estatísticas com top 10

N = 10

import pandas as pd

def summarize_models(results, TARGETS, SETS, N=10, output="Resumo_Estatisticas.xlsx"):
    
    best_models_tables = {}
    best_only = []
    summary_all = []
    summary_top10 = []

    for target in TARGETS:
        for dataset in SETS:

            col_r2 = f"R2_{dataset}_{target}"
            col_mse = f"MSE_{dataset}_{target}"

            if col_r2 not in results.columns:
                continue

            table = results.copy()

            table = table.sort_values(by=col_r2, ascending=False)

            best_models_tables[f"{dataset}_{target}"] = table

            # 🔹 melhor modelo
            best = table.iloc[0]

            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": best["model"],
                "Neurons": best["Neurons"],
                "Best_R2": best[col_r2]
            })

            # 🔹 estatísticas de todos
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })

            # 🔹 estatísticas top N
            topN = table.head(N)

            summary_topN.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": topN[col_r2].mean(),
                "std_r2": topN[col_r2].std(),
                "mean_mse": topN[col_mse].mean(),
                "std_mse": topN[col_mse].std()
            })

    df_summary_all = pd.DataFrame(summary_all)
    df_summary_topN = pd.DataFrame(summary_topN)
    best_only_df = pd.DataFrame(best_only)
    # exportar
    with pd.ExcelWriter(output) as writer:
        df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
        df_summary_topN.to_excel(writer, sheet_name="Top_Modelos", index=False)
        best_only_df.to_excel(writer, sheet_name="Melhor_Modelo", index=False)

    return best_models_tables, df_summary_all, df_summary_topN, best_only_df

In [41]:
tables, summary_all, summary_top10, best_models = summarize_models(
    results, TARGETS, SETS
)

In [43]:
summary_top10["bestR2"] = best_models["Best_R2"]
summary_top10["Neurons"] = best_models["Neurons"]
display(summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,Wd,0.937368,0.007030,0.330404,0.037088,0.948163,[264]
1,Train_2,Wd,0.923535,0.006407,0.402242,0.033702,0.937418,[128]
2,Val,Wd,0.891270,0.012519,0.461227,0.053107,0.912051,"[32, 16, 8]"
3,Test_1,Wd,0.926997,0.005707,0.378977,0.029627,0.937616,[128]
4,Test_2,Wd,0.931125,0.008049,0.354328,0.041406,0.949595,[128]
5,Test_3,Wd,0.871855,0.010427,0.477498,0.038855,0.886415,[128]
6,Train_1,We,0.929272,0.004499,0.384481,0.024456,0.937883,[264]
7,Train_2,We,0.937237,0.005176,0.341057,0.028124,0.946808,[264]
8,Val,We,0.889446,0.007165,0.466141,0.030212,0.902435,"[8, 4]"
9,Test_1,We,0.925616,0.003773,0.388205,0.019693,0.932820,[264]


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted